In [8]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Supported:", torch.cuda.get_arch_list())

x = torch.randn(100, 100, device="cuda")
print((x @ x).shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

PyTorch: 2.7.1+cu118
CUDA: 11.8
GPU: NVIDIA GeForce GTX 1060
Supported: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_37', 'sm_90']
torch.Size([100, 100])
cuda


In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

base = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/titanic/"
def load(name):
    path = Path("dataset/titanic") / name
    return pd.read_csv(path if path.exists() else base + name)

train, test = load("train.csv"), load("test.csv")
torch.manual_seed(42)

In [10]:
def features(df):
    x = df[["Age", "SibSp", "Parch", "Fare", "Pclass", "Sex", "Embarked"]].copy()
    title = df["Name"].str.extract(r",\s*([^.]*)\.")[0].str.strip()
    x["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    x["HasCabin"] = df["Cabin"].notna().astype(int)
    return x

num = ["Age", "SibSp", "Parch", "Fare", "HasCabin"]
cat = ["Pclass", "Sex", "Embarked", "Title"]

def fit(X, y, hidden=32, dropout=0.0, lr=0.01, weight_decay=0.0, epochs=200):
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat),
    ])

    x = torch.tensor(prep.fit_transform(X), dtype=torch.float32, device=device)
    y = torch.tensor(y.to_numpy(), dtype=torch.float32, device=device).view(-1, 1)

    model = nn.Sequential(
        nn.Linear(x.shape[1], hidden),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, 1)
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    return model, prep

def predict(model, prep, X):
    x = torch.tensor(prep.transform(X), dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        return (model(x).sigmoid().squeeze(1) >= 0.5).cpu().numpy().astype(int)

In [11]:
X, y = features(train), train["Survived"]

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
import optuna

def objective(trial):
    params = {
        "hidden": trial.suggest_categorical("hidden", [16, 32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.4),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "epochs": trial.suggest_int("epochs", 80, 300, step=20),
    }

    torch.manual_seed(42)
    model, prep = fit(X_tr, y_tr, **params)

    return (predict(model, prep, X_val) == y_val.to_numpy()).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print(study.best_value)
print(study.best_params)

[I 2026-10-12 11:25:00,138] A new study created in memory with name: no-name-cbcd866d-5794-48b7-a762-3b3b73bfcb59
[I 2026-10-12 11:25:00,577] Trial 0 finished with value: 0.7821229050279329 and parameters: {'hidden': 128, 'dropout': 0.15973309056170326, 'lr': 0.004931132441344491, 'weight_decay': 0.0004587585261510391, 'epochs': 280}. Best is trial 0 with value: 0.7821229050279329.
[I 2026-10-12 11:25:00,753] Trial 1 finished with value: 0.776536312849162 and parameters: {'hidden': 32, 'dropout': 0.14892973624923944, 'lr': 0.001010945082014957, 'weight_decay': 0.00042412889905641185, 'epochs': 100}. Best is trial 0 with value: 0.7821229050279329.
[I 2026-10-12 11:25:01,168] Trial 2 finished with value: 0.8044692737430168 and parameters: {'hidden': 32, 'dropout': 0.2082835886553984, 'lr': 0.0006175242889633877, 'weight_decay': 4.6114271559583855e-06, 'epochs': 260}. Best is trial 2 with value: 0.8044692737430168.
[I 2026-10-12 11:25:01,519] Trial 3 finished with value: 0.765363128491620

0.8379888268156425
{'hidden': 128, 'dropout': 0.13184433163796416, 'lr': 0.0009870555677357613, 'weight_decay': 0.0018013586619662228, 'epochs': 160}


In [13]:
X, y = features(train), train["Survived"]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model, prep = fit(X_tr, y_tr)
accuracy = (predict(model, prep, X_val) == y_val.to_numpy()).mean()
print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8212


In [14]:
model, prep = fit(X, y)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predict(model, prep, features(test)),
})
submission.to_csv("submission_nearal_network.csv", index=False)
print("Saved submission")
submission.head()

Saved submission


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
